# Offline d1/d2/d3 validation harness

Scores any retrieval method on synthetic proxies of all three difficulty levels,
built from dataset1's labelled pairs. **Run All** and read the table at the bottom.

- Lives in `amine/` alongside `eval_harness.py` (kernel cwd = this folder).
- Cell 2 symlinks `data/` and `out/` here so the data + outputs show in the left panel.
- Edit `EMBEDDERS` in `eval_harness.py` to plug in new methods, then re-run.
- Last cell gives download links for the outputs.

In [1]:
import subprocess, sys
# nibabel+scipy for the reference embedders; torch+monai for the learned/Swin embedder.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nibabel>=5.3', 'scipy', 'monai>=1.3'], check=True)
print('deps ready')

deps ready


In [2]:
import os
# Make the input data and output dir browsable in the file panel on the left:
# create symlinks INSIDE this folder (amine/) pointing at the real locations.
# One-time per container; harmless if they already exist.
for name, target in [('data', '/workspace/data/ehl'), ('out', '/workspace/out')]:
    if os.path.islink(name) or os.path.exists(name):
        print('exists:', name, '->', os.path.realpath(name))
        continue
    if os.path.isdir(target):
        os.symlink(target, name)
        print('linked:', name, '->', target)
    else:
        print('SKIP', name, '- target not found:', target, '(fix the path if your mount differs)')

exists: data -> /workspace/data/ehl
exists: out -> /workspace/out


In [3]:
import os
# The box mounts the data here (see CLAUDE_workflow.md). Adjust if your mount differs.
os.environ['DATA_ROOT'] = '/workspace/data/ehl'
os.environ.setdefault('N_VAL', '60')   # held-out dataset1 pairs to score on
os.environ.setdefault('GRID', '96')    # resample cube size

# --- FAST PASS: classical GPU rankers only (NMI / MIND / gradient cosine) -----
# These need NO training, so this pass is quick and is the highest-ROI signal:
#   nmi      -> dataset1 (aligned, common grid) — expected the big win
#   mind     -> dataset2/3 (deformation- & modality-tolerant)
#   nmi_grad -> robust blend for dataset1
# Restrict which rankers run with e.g. RANKERS='nmi,mind'; default runs all four.
os.environ['SKIP_LEARNED'] = '1'        # set to '0' to also train the learned/Swin embedder
# os.environ['RANKERS'] = 'nmi,mind,nmi_grad,gradcos'

# To additionally train the learned embedder, set SKIP_LEARNED='0' and pick a backbone:
# os.environ['EMB_BACKBONE'] = 'swin'   # 'cnn' (fast) or 'swin' (SSL-pretrained, GRID%32==0)

assert os.path.isdir(os.environ['DATA_ROOT']), 'data root not found - fix DATA_ROOT above'
assert os.path.exists('eval_harness.py'), 're-upload eval_harness.py + rankers.py here'
assert os.path.exists('rankers.py'), 're-upload rankers.py here'
print('using', os.environ['DATA_ROOT'], '| skip_learned', os.environ['SKIP_LEARNED'])

using /workspace/data/ehl | skip_learned 1


In [ ]:
import importlib, eval_harness
importlib.reload(eval_harness)   # pick up edits without restarting the kernel
results = eval_harness.evaluate()

DATA_ROOT=/workspace/data/ehl  indexed=1454  val_pairs=60  train_pairs=290  grid=96
[mind] device=cuda pairs=290 cfg={'epochs': 200, 'batch': 64, 'dim': 128, 'width': 24, 'lr': 0.0003, 'rot': 20.0, 'elastic': 0.08, 'resect': 0.5, 'wd': 0.01, 'dil': 2, 'tta': 8}
[mind] epoch 001 loss=4.0796 (152s)


In [ ]:
import os, shutil
from IPython.display import FileLink, display
# Click these links to download outputs to your computer.
# (submission.csv is produced by run_baseline.ipynb; searched in a few common spots.)
def offer(fname, candidates):
    for c in candidates:
        if os.path.exists(c):
            if os.path.abspath(c) != os.path.abspath(fname):
                shutil.copy(c, fname)   # bring it next to this notebook so the link resolves
            display(FileLink(fname))
            return
    print('not found yet:', fname)

offer('eval_results.md', ['eval_results.md'])
offer('submission.csv', ['submission.csv', 'out/submission.csv',
                         '/workspace/out/submission.csv', '/root/submission.csv'])

In [ ]:
from IPython.display import Markdown
Markdown(open('eval_results.md').read())